### University of Virginia
### DS 5110: Big Data Systems

### Lab: Supervised Learning
### Last updated: March 2, 2023

---

#### Instructions

This project has two parts:
- Part I: Classification - build and apply a logistic regression model on the Wisconsin Breast Cancer dataset.
- Part II: Regression - build and apply a linear regression model on the California Housing dataset.

**Total Possible Points: 10**

---

#### Part I: Classification (5 POINTS)

Here are the specifications and grading breakdown:

- the target variable is `diagnosis`
- use `f1`, `f2` as predictors (1 PT)
- split data into 60% training set, 40% test set 
- standardize the predictors (1 PT)
- use seed=314 whenever a seed is needed
- fit a Logistic Regression model with an intercept (1 PT)
- compute and show the area under the ROC curve for the test set (2 PTS)

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [2]:
DATA_FILEPATH = 'wisc_breast_cancer_w_fields.csv'

spark = SparkSession \
    .builder \
    .appName("Wisc BRCA") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/09 15:43:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/09 15:43:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


#### Enter code and solution

In [3]:
# Load and view data
df = spark.read.csv(DATA_FILEPATH, inferSchema=True, header=True)
df.printSchema()
df.select("diagnosis", "f1", "f2").show(5)
df.groupBy("diagnosis").count().show()

root
 |-- id: integer (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- f1: double (nullable = true)
 |-- f2: double (nullable = true)
 |-- f3: double (nullable = true)
 |-- f4: double (nullable = true)
 |-- f5: double (nullable = true)
 |-- f6: double (nullable = true)
 |-- f7: double (nullable = true)
 |-- f8: double (nullable = true)
 |-- f9: double (nullable = true)
 |-- f10: double (nullable = true)
 |-- f11: double (nullable = true)
 |-- f12: double (nullable = true)
 |-- f13: double (nullable = true)
 |-- f14: double (nullable = true)
 |-- f15: double (nullable = true)
 |-- f16: double (nullable = true)
 |-- f17: double (nullable = true)
 |-- f18: double (nullable = true)
 |-- f19: double (nullable = true)
 |-- f20: double (nullable = true)
 |-- f21: double (nullable = true)
 |-- f22: double (nullable = true)
 |-- f23: double (nullable = true)
 |-- f24: double (nullable = true)
 |-- f25: double (nullable = true)
 |-- f26: double (nullable = true)
 |-- f27: double (

In [4]:
# Convert response to index
indexer = StringIndexer(inputCol="diagnosis", outputCol="label")
df = indexer.fit(df).transform(df)

df.groupBy("diagnosis", "label").count().show()

+---------+-----+-----+
|diagnosis|label|count|
+---------+-----+-----+
|        B|  0.0|  357|
|        M|  1.0|  212|
+---------+-----+-----+



In [5]:
# Split data
train, test = df.randomSplit([0.6, 0.4], seed=314)

In [6]:
# Assemble features (both sets)
assembler = VectorAssembler(inputCols=["f1", "f2"], outputCol="features")
train = assembler.transform(train)
test = assembler.transform(test)

In [7]:
# Fit standardization on training set and apply to both
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures",
                        withMean=True, withStd=True)
scalerModel = scaler.fit(train)
train = scalerModel.transform(train)
test = scalerModel.transform(test)

26/10/09 15:43:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [8]:
# Fit logistic regression
lr = LogisticRegression(labelCol="label", featuresCol="scaledFeatures", fitIntercept=True)
lrModel = lr.fit(train)

26/10/09 15:43:15 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [9]:
# Predict with test set
preds = lrModel.transform(test)
preds.select("label", "rawPrediction", "probability", "prediction").show(5, truncate=False)

+-----+----------------------------------------+------------------------------------------+----------+
|label|rawPrediction                           |probability                               |prediction|
+-----+----------------------------------------+------------------------------------------+----------+
|0.0  |[2.4982279599735944,-2.4982279599735944]|[0.9240174999811224,0.07598250001887763]  |0.0       |
|1.0  |[-6.54413540541027,6.54413540541027]    |[0.0014364608997719049,0.9985635391002281]|1.0       |
|0.0  |[1.6495572730639514,-1.6495572730639514]|[0.8388312056356981,0.16116879436430187]  |0.0       |
|0.0  |[0.3016230916524252,-0.3016230916524252]|[0.5748392470339003,0.4251607529660997]   |0.0       |
|0.0  |[1.3268721720726835,-1.3268721720726835]|[0.790322784707739,0.20967721529226102]   |0.0       |
+-----+----------------------------------------+------------------------------------------+----------+
only showing top 5 rows


In [10]:
# Compute AUC
evaluator = BinaryClassificationEvaluator(labelCol="label",
                                          rawPredictionCol="rawPrediction",
                                          metricName="areaUnderROC")
auc = evaluator.evaluate(preds)
print("Test AUC:", auc)

Test AUC: 0.9514088556641738


#### Part II: Regression (5 POINTS)

In this project, you will work with the California Home Price dataset to train a regression model and predict median home prices. Here are the specifications and grading breakdown:

- Scale the response variable median_house_value, dividing by 100000 (1 PT)

- Split data into train set (80%), test set (20%) using seed=314 (1 PT)

- Add new predictor: `rooms_per_household`

- In the training set, select all of these features and standardize them: (1 PT)

feats = ["total_bedrooms", 
         "population", 
         "households", 
         "median_income", 
         "rooms_per_household"]

- Fit a linear regression model on the training set with these parameters:

  - maxIter=10
  - regParam=0.3
  - elasticNetParam=0.8  


- Compute the MSE on the test set (2 PTS)

In [11]:
import os
import pandas as pd
from pyspark.sql.functions import col
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [12]:
spark = SparkSession.builder.getOrCreate()

In [13]:
DATA_FILEPATH2 = 'cal_housing_data_preproc_w_header.txt'

#### Enter code and solution

In [14]:
# Load and view data
df2 = spark.read.csv(DATA_FILEPATH2, inferSchema=True, header=True)
df2.printSchema()
df2.show(3)

root
 |-- median_house_value: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)

+------------------+-----------------+------------------+-----------+--------------+----------+----------+--------+---------+
|median_house_value|    median_income|housing_median_age|total_rooms|total_bedrooms|population|households|latitude|longitude|
+------------------+-----------------+------------------+-----------+--------------+----------+----------+--------+---------+
|          452600.0|           8.3252|              41.0|      880.0|         129.0|     322.0|     126.0|   37.88|  -122.23|
|          358500.0|           8.3014|              21.0|     7099.0|        1106.0|    240

In [15]:
# Transform columns
df2 = df2.withColumn("median_house_value", col("median_house_value") / 1e5)
df2 = df2.withColumn("rooms_per_household", col("total_rooms") / col("households"))

In [16]:
# Split data
train2, test2 = df2.randomSplit([0.8, 0.2], seed=314)

In [17]:
# Assemble predictors (both sets)
feats = ["total_bedrooms", "population", "households", "median_income", "rooms_per_household"]
assembler2 = VectorAssembler(inputCols=feats, outputCol="features")
train2 = assembler2.transform(train2)
test2 = assembler2.transform(test2)

In [18]:
# Fit standardization on training set and apply to both
scaler2 = StandardScaler(inputCol="features", outputCol="scaledFeatures",
                         withMean=True, withStd=True)
scalerModel2 = scaler2.fit(train2)
train2 = scalerModel2.transform(train2)
test2 = scalerModel2.transform(test2)

In [19]:
# Fit linear regression
lr2 = LinearRegression(featuresCol="scaledFeatures", labelCol="median_house_value",
                       maxIter=10, regParam=0.3, elasticNetParam=0.8)
lrModel2 = lr2.fit(train2)

In [20]:
# Predict with test set and calculate MSE
preds2 = lrModel2.transform(test2)
evaluator2 = RegressionEvaluator(labelCol="median_house_value", predictionCol="prediction",
                                 metricName="mse")
mse = evaluator2.evaluate(preds2)
print("Test MSE:", mse)

Test MSE: 0.7551749809614869
